---
### import 패키지

In [17]:
import pandas as pd
import numpy as np
import os
import ollama
from pydantic import BaseModel, Field
from google import genai

---
### 전역변수

In [18]:
MODEL = "bge-m3"
BATCH = 100
N_PATH = "/Users/d.h/dev/ai_chatbot/data_file/netflix_titles.csv"
B_PATH = "/Users/d.h/dev/ai_chatbot/data_file/netflix_desctiption.npy"

---
### description 벡터화

In [19]:
df = pd.read_csv(N_PATH)
df = df.fillna("Unknown")
extract_description = df["description"].tolist()
#extract_description = extract_description.tolist()
extract_show_id = df["show_id"]

In [20]:
def embed(extract_description):
    vectors = []
    for i in range(0, len(extract_description), BATCH):
        batch = extract_description[i:i + BATCH]
        result = ollama.embed(model=MODEL, input=batch)
        vectors.extend(result["embeddings"])
        if len(extract_description) > BATCH:
            print(f"임베딩 {len(vectors)}/{len(extract_description)}")
    vectors = np.array(vectors, dtype=np.float32)
    return vectors / np.linalg.norm(vectors, axis=1, keepdims=True)

---
### 한번만 벡터화 하기

In [23]:
def load_or_build(extract_description, B_PATH):
    if os.path.exists(B_PATH):
        saved = np.load(B_PATH)
        if len(saved) == len(extract_description):
            print("저장된 벡터 불러옴:", saved.shape)
            return saved
        print("문서수가 달라서 다시 만듭니다.")
    vectors = embed(extract_description)
    np.save(B_PATH, vectors)
    print("저장 완료:", vectors.shape)
    return vectors

In [24]:
if __name__ == "__main__":
    load_or_build(extract_description, B_PATH)

저장된 벡터 불러옴: (8807, 1024)
